In [ ]:
!pip install yfinance, numpy, pandas, Matplotlib

ERROR: Invalid requirement: 'yfinance,': Expected end or semicolon (after name and no valid version specifier)
    yfinance,
            ^


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

In [ ]:
from datetime import datetime, timedelta
import yfinance as yf

today = datetime.now()
one_year_ago = today - timedelta(days=1*365) #Time delta helps to find time difference between times.
stock_ticker = input("Enter the stock_ticker for the company you are thinking about?")
company_data = yf.download(stock_ticker, start = one_year_ago.strftime('%Y-%m-%d'), end = today.strftime('%Y-%m-%d'))

Enter the stock_ticker for the company you are thinking about?AAPL


/tmp/ipykernel_1584/812111838.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  company_data = yf.download(stock_ticker, start = one_year_ago.strftime('%Y-%m-%d'), end = today.strftime('%Y-%m-%d'))
[*********************100%***********************]  1 of 1 completed


In [ ]:
#Basic Statistics
print(company_data.head())

# Correctly access the 'Close' price Series for the given ticker
close_prices = company_data[('Close', stock_ticker)]

max_close_price = np.max(close_prices)
min_close_price = np.min(close_prices)

# idxmax() on a Series with DatetimeIndex returns a Timestamp, which has strftime
max_close_date = close_prices.idxmax().strftime('%Y-%m-%d')
min_close_date = close_prices.idxmin().strftime('%Y-%m-%d')

print(f"Average Closing Price: {np.mean(close_prices):.2f}")
print(f"Maximum Closing Price: {max_close_price:.2f} on {max_close_date}")
print(f"Minimum Closing Price: {min_close_price:.2f} on {min_close_date}")

# Calculate Total Return Percentage (Concise version)
initial_close_price = close_prices.iloc[0]
# Get the last valid close price by dropping NaNs first
final_close_price = close_prices.dropna().iloc[-1]

# Check for division by zero before calculating percentage
if initial_close_price == 0:
    print("Cannot calculate Total Return Percentage: Initial closing price is zero.")
elif pd.isna(initial_close_price) or pd.isna(final_close_price):
    print("Cannot calculate Total Return Percentage: Missing initial or final close price.")
else:
    total_return_percentage = ((final_close_price - initial_close_price) / initial_close_price) * 100
    print(f"Total Return Percentage: {total_return_percentage:.2f}%")

Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2025-07-25  213.034851  214.389478  212.556737  213.851603  40268800
2025-07-28  213.204178  214.001020  212.218084  213.184253  37858000
2025-07-29  210.435150  213.961155  209.986931  213.333639  51411700
2025-07-30  208.223923  211.550721  206.899177  211.062652  45512500
2025-07-31  206.749786  209.010805  206.341403  207.666149  80698400
Average Closing Price: 266.63
Maximum Closing Price: 333.74 on 2026-07-17
Minimum Closing Price: 201.58 on 2025-08-01
Total Return Percentage: 50.99%


In [ ]:
daily_returns = close_prices.pct_change().dropna()

daily_std_dev = np.std(daily_returns)
print(f"Daily Returns Standard Deviation (Risk): {daily_std_dev:.4f}")

Daily Returns Standard Deviation (Risk): 0.0155


/tmp/ipykernel_1584/902895012.py:1: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  daily_returns = close_prices.pct_change().dropna()


In [ ]:
# Assuming a risk-free rate of 0% for simplicity in this example (often 0.02 for annual)
risk_free_rate = 0 # Daily risk-free rate, assuming 0 for simplicity

# Calculate average daily return
average_daily_return = np.mean(daily_returns)

# Calculate Sharpe Ratio
# Sharpe Ratio = (Average Daily Return - Risk-Free Rate) / Daily Standard Deviation
if daily_std_dev == 0:
    sharpe_ratio = np.nan # Avoid division by zero
else:
    sharpe_ratio = (average_daily_return - risk_free_rate) / daily_std_dev

print(f"Sharpe Ratio: {sharpe_ratio:.4f}")

Sharpe Ratio: 0.1142


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Price over time
axes[0].plot(close_prices.index, close_prices.values, color='blue')
axes[0].set_title(f'{stock_ticker} Stock Price Over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Closing Price (USD)')
axes[0].grid(True)

# Plot 2: Histogram of daily returns
axes[1].hist(daily_returns, bins=50, color='green', edgecolor='black')
axes[1].set_title('Histogram of Daily Returns')
axes[1].set_xlabel('Daily Return')
axes[1].set_ylabel('Frequency')
axes[1].grid(True)

plt.tight_layout()
plt.show()